# Bitcoin Fiyatlarının Makroekonomik Faktörlerle Analizi

Bu notebook, Bitcoin fiyatlarını makroekonomik faktörlerle analiz eder ve makine öğrenmesi modelleri eğitir.


## 1. Kütüphanelerin İçe Aktarılması


In [ ]:
from __future__ import annotations

import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# Jupyter için görselleştirme ayarları
%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Kütüphaneler başarıyla yüklendi!")


## 2. Veri Yükleme ve Hazırlama


In [ ]:
# Yol tanımlamaları
BASE_DIR = Path('.').resolve()
PROCESSED_DIR = BASE_DIR / "processed"
DEFAULT_MERGED = PROCESSED_DIR / "merged_data.csv"
HEATMAP_PATH = PROCESSED_DIR / "correlation_heatmap.png"

# Veri yükleme
def load_dataset(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=["Date"])
    if df["Date"].dtype != "datetime64[ns]":
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"])
    return df

# Veri setini yükle
if DEFAULT_MERGED.exists():
    df = load_dataset(DEFAULT_MERGED)
    print(f"Veri yüklendi: {df.shape[0]} satır, {df.shape[1]} sütun")
    print(f"Tarih aralığı: {df['Date'].min().date()} → {df['Date'].max().date()}")
else:
    print(f"HATA: {DEFAULT_MERGED} bulunamadı!")
    print("Önce preprocess.py dosyasını çalıştırın.")
    raise FileNotFoundError(f"{DEFAULT_MERGED} bulunamadı")


## 3. Veri Önizleme


In [ ]:
# Veri önizleme
print("\nİlk 5 satır:")
display(df.head())

print("\nÖzet İstatistikler:")
display(df.select_dtypes(include='number').describe())

print(f"\nToplam gözlem: {len(df):,}")
print(f"Eksik değerler:")
missing = df.isnull().sum()
print(missing[missing > 0])


## 4. Model Parametrelerinin Ayarlanması

Aşağıdaki parametreleri kendi tercihlerinize göre değiştirebilirsiniz:


In [ ]:
# ============================================
# MODEL SEÇİMİ VE PARAMETRELER
# ============================================

# Seçilecek modeller (2 adet seçin)
SELECTED_MODELS = ["LinearRegression", "DecisionTree"]  # Veya sadece birini seçin

# Linear Regression Parametreleri
LR_FIT_INTERCEPT = True

# Decision Tree Parametreleri
DT_MAX_DEPTH = 10  # None, 3, 5, 10, 15, 20, 30
DT_MIN_SAMPLES_SPLIT = 2  # 2, 5, 10, 20
DT_MIN_SAMPLES_LEAF = 1  # 1, 2, 4, 5
DT_CRITERION = "squared_error"  # "squared_error", "friedman_mse", "absolute_error", "poisson"

# Feature Selection Parametreleri
N_FEATURES_SELECT = 10  # Mutual Information ile seçilecek özellik sayısı
N_COMPONENTS_PCA = 5  # PCA bileşen sayısı

# Train-Test Split
TEST_SIZE = 0.2  # Test set oranı (0.1 - 0.4 arası)
RANDOM_STATE = 42

print("Parametreler ayarlandı:")
print(f"  - Seçilen Modeller: {SELECTED_MODELS}")
print(f"  - Test Set Oranı: {TEST_SIZE}")
print(f"  - Feature Selection: {N_FEATURES_SELECT} özellik")
print(f"  - PCA Bileşen Sayısı: {N_COMPONENTS_PCA}")


## 5. Hedef Değişkenlerin Oluşturulması


In [ ]:
# Veriyi tarihe göre sırala
ml_df = df.copy().sort_values("Date").reset_index(drop=True)
target_col = "btc_close"

if target_col not in ml_df.columns:
    raise ValueError(f"Hedef değişken '{target_col}' bulunamadı!")

# Hedef değişkenleri oluştur (gelecekteki fiyatlar)
ml_df["Target_1d"] = ml_df[target_col].shift(-1)
ml_df["Target_7d"] = ml_df[target_col].shift(-7)
ml_df["Target_30d"] = ml_df[target_col].shift(-30)
ml_df["Target_365d"] = ml_df[target_col].shift(-365)

target_vars = ["Target_1d", "Target_7d", "Target_30d", "Target_365d"]
available_targets = [t for t in target_vars if ml_df[t].notna().sum() > 0]

print(f"Oluşturulan hedef değişkenler: {', '.join(available_targets)}")
for target in available_targets:
    count = ml_df[target].notna().sum()
    print(f"  - {target}: {count} geçerli gözlem")


## 6. Özellik Hazırlama


In [ ]:
# Özellik sütunlarını hazırla
numeric_cols = ml_df.select_dtypes(include=[np.number]).columns.tolist()
for col in [target_col] + target_vars + ["Date"]:
    if col in numeric_cols:
        numeric_cols.remove(col)

print(f"Toplam özellik sayısı: {len(numeric_cols)}")
print(f"\nÖzellikler:")
for i, col in enumerate(numeric_cols, 1):
    print(f"  {i}. {col}")


## 7. Model Eğitimi ve Değerlendirme


In [ ]:
# Model sınıfları
available_models = {
    "LinearRegression": LinearRegression,
    "DecisionTree": DecisionTreeRegressor
}

# Model parametreleri
model_params = {}
if "LinearRegression" in SELECTED_MODELS:
    model_params["LinearRegression"] = {"fit_intercept": LR_FIT_INTERCEPT}

if "DecisionTree" in SELECTED_MODELS:
    model_params["DecisionTree"] = {
        "max_depth": DT_MAX_DEPTH,
        "min_samples_split": DT_MIN_SAMPLES_SPLIT,
        "min_samples_leaf": DT_MIN_SAMPLES_LEAF,
        "criterion": DT_CRITERION,
        "random_state": RANDOM_STATE
    }

# Direction Accuracy hesaplama fonksiyonu
def direction_accuracy(y_true, y_pred):
    """Yön tahmini doğruluğunu hesapla (artış/azalış)"""
    if len(y_true) < 2:
        return 0.0
    true_direction = np.diff(y_true) > 0
    pred_direction = np.diff(y_pred) > 0
    return np.mean(true_direction == pred_direction)

# Tüm sonuçları saklamak için liste
all_results = []

print("\n" + "="*60)
print("MODEL EĞİTİMİ BAŞLIYOR")
print("="*60)

total_experiments = len(SELECTED_MODELS) * len(available_targets) * 3  # 3 feature scenario
current_exp = 0

for target_var in available_targets:
    print(f"\n{'='*60}")
    print(f"HEDEF: {target_var}")
    print(f"{'='*60}")
    
    # Bu hedef için veriyi hazırla
    X_full = ml_df[numeric_cols].copy()
    y_full = ml_df[target_var].copy()
    
    # NaN değerleri temizle
    mask = ~(X_full.isna().any(axis=1) | y_full.isna())
    X_full = X_full[mask].reset_index(drop=True)
    y_full = y_full[mask].reset_index(drop=True)
    
    if len(X_full) < 10:
        print(f"  ⚠️  Yeterli veri yok, atlanıyor...")
        continue
    
    # Train-test split (zaman serisi için shuffle=False)
    split_idx = int(len(X_full) * (1 - TEST_SIZE))
    X_train_full = X_full.iloc[:split_idx]
    X_test_full = X_full.iloc[split_idx:]
    y_train_full = y_full.iloc[:split_idx]
    y_test_full = y_full.iloc[split_idx:]
    
    print(f"  Eğitim seti: {len(X_train_full)} gözlem")
    print(f"  Test seti: {len(X_test_full)} gözlem")
    
    # Özellik senaryoları
    feature_scenarios = {}
    
    # 1. Full Features
    feature_scenarios["Full Features"] = {
        "X_train": X_train_full,
        "X_test": X_test_full
    }
    
    # 2. Feature Selection (Mutual Information)
    if len(X_train_full.columns) > 1:
        print(f"  \n  🔍 Feature Selection hesaplanıyor...")
        mi_scores = mutual_info_regression(X_train_full, y_train_full, random_state=RANDOM_STATE)
        feature_importance_df = pd.DataFrame({
            "Feature": X_train_full.columns,
            "MI_Score": mi_scores
        }).sort_values("MI_Score", ascending=False)
        
        selected_features = feature_importance_df.head(N_FEATURES_SELECT)["Feature"].tolist()
        feature_scenarios["Feature Selection"] = {
            "X_train": X_train_full[selected_features],
            "X_test": X_test_full[selected_features]
        }
        print(f"     Seçilen özellikler: {', '.join(selected_features[:5])}...")
    
    # 3. PCA
    print(f"  \n  📊 PCA uygulanıyor...")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_full)
    X_test_scaled = scaler.transform(X_test_full)
    
    pca = PCA(n_components=min(N_COMPONENTS_PCA, X_train_scaled.shape[1]))
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    
    feature_scenarios["PCA"] = {
        "X_train": pd.DataFrame(X_train_pca),
        "X_test": pd.DataFrame(X_test_pca)
    }
    print(f"     PCA açıklanan varyans: {pca.explained_variance_ratio_.sum():.4f}")
    
    # Her senaryo için model eğitimi
    for scenario_name, scenario_data in feature_scenarios.items():
        print(f"\n  📈 Senaryo: {scenario_name}")
        
        X_train = scenario_data["X_train"]
        X_test = scenario_data["X_test"]
        
        for model_name in SELECTED_MODELS:
            current_exp += 1
            print(f"    [{current_exp}/{total_experiments}] {model_name} eğitiliyor...", end=" ")
            
            # Model oluştur ve eğit
            model_class = available_models[model_name]
            params = model_params[model_name].copy()
            model = model_class(**params)
            
            model.fit(X_train, y_train_full)
            
            # Tahminler
            y_pred_train = model.predict(X_train)
            y_pred_test = model.predict(X_test)
            
            # Metrikleri hesapla
            rmse_train = np.sqrt(mean_squared_error(y_train_full, y_pred_train))
            rmse_test = np.sqrt(mean_squared_error(y_test_full, y_pred_test))
            mae_train = mean_absolute_error(y_train_full, y_pred_train)
            mae_test = mean_absolute_error(y_test_full, y_pred_test)
            r2_train = r2_score(y_train_full, y_pred_train)
            r2_test = r2_score(y_test_full, y_pred_test)
            dir_acc_train = direction_accuracy(y_train_full.values, y_pred_train)
            dir_acc_test = direction_accuracy(y_test_full.values, y_pred_test)
            
            # Sonuçları kaydet
            result = {
                "Model": model_name,
                "Feature Set": scenario_name,
                "Target": target_var,
                "Parameters": str(params),
                "RMSE (Train)": rmse_train,
                "RMSE (Test)": rmse_test,
                "MAE (Train)": mae_train,
                "MAE (Test)": mae_test,
                "R² (Train)": r2_train,
                "R² (Test)": r2_test,
                "Direction Accuracy (Train)": dir_acc_train,
                "Direction Accuracy (Test)": dir_acc_test
            }
            all_results.append(result)
            
            print(f"✓ R² Test: {r2_test:.4f}, RMSE Test: {rmse_test:.4f}")

print("\n" + "="*60)
print("TÜM DENEMELER TAMAMLANDI!")
print("="*60)


In [ ]:
if all_results:
    # Sonuçları DataFrame'e dönüştür
    results_df = pd.DataFrame(all_results)
    
    # Sütun sırasını düzenle
    column_order = [
        "Model", "Feature Set", "Target", "Parameters",
        "RMSE (Test)", "MAE (Test)", "R² (Test)", "Direction Accuracy (Test)",
        "RMSE (Train)", "MAE (Train)", "R² (Train)", "Direction Accuracy (Train)"
    ]
    results_df = results_df[column_order]
    
    # Sonuçları göster
    print("\n" + "="*80)
    print("SONUÇLAR TABLOSU")
    print("="*80)
    display(results_df.style.format({
        "RMSE (Train)": "{:.4f}",
        "RMSE (Test)": "{:.4f}",
        "MAE (Train)": "{:.4f}",
        "MAE (Test)": "{:.4f}",
        "R² (Train)": "{:.4f}",
        "R² (Test)": "{:.4f}",
        "Direction Accuracy (Train)": "{:.4f}",
        "Direction Accuracy (Test)": "{:.4f}"
    }))
    
    # Özet istatistikler
    print("\n" + "="*80)
    print("ÖZET İSTATİSTİKLER")
    print("="*80)
    
    best_rmse = results_df.loc[results_df["RMSE (Test)"].idxmin()]
    best_mae = results_df.loc[results_df["MAE (Test)"].idxmin()]
    best_r2 = results_df.loc[results_df["R² (Test)"].idxmax()]
    best_dir_acc = results_df.loc[results_df["Direction Accuracy (Test)"].idxmax()]
    
    print(f"\n🏆 En İyi RMSE (Test): {best_rmse['RMSE (Test)']:.4f}")
    print(f"   Model: {best_rmse['Model']}, Feature Set: {best_rmse['Feature Set']}, Target: {best_rmse['Target']}")
    
    print(f"\n🏆 En İyi MAE (Test): {best_mae['MAE (Test)']:.4f}")
    print(f"   Model: {best_mae['Model']}, Feature Set: {best_mae['Feature Set']}, Target: {best_mae['Target']}")
    
    print(f"\n🏆 En İyi R² (Test): {best_r2['R² (Test)']:.4f}")
    print(f"   Model: {best_r2['Model']}, Feature Set: {best_r2['Feature Set']}, Target: {best_r2['Target']}")
    
    print(f"\n🏆 En İyi Direction Accuracy (Test): {best_dir_acc['Direction Accuracy (Test)']:.4f}")
    print(f"   Model: {best_dir_acc['Model']}, Feature Set: {best_dir_acc['Feature Set']}, Target: {best_dir_acc['Target']}")
    
else:
    print("⚠️  Sonuç bulunamadı!")


In [ ]:
if all_results:
    # CSV olarak kaydet
    output_path = PROCESSED_DIR / "model_results.csv"
    results_df.to_csv(output_path, index=False)
    print(f"\n✅ Sonuçlar kaydedildi: {output_path}")
    
    # Excel formatında da kaydet (opsiyonel)
    try:
        excel_path = PROCESSED_DIR / "model_results.xlsx"
        results_df.to_excel(excel_path, index=False)
        print(f"✅ Excel formatında kaydedildi: {excel_path}")
    except ImportError:
        print("⚠️  Excel kaydetmek için 'openpyxl' paketi gerekli: pip install openpyxl")
else:
    print("⚠️  Kaydedilecek sonuç yok!")


## 10. Görselleştirmeler (Opsiyonel)


In [ ]:
if all_results:
    # Model performans karşılaştırması
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # RMSE karşılaştırması
    ax1 = axes[0, 0]
    pivot_rmse = results_df.pivot_table(
        values='RMSE (Test)', 
        index=['Model', 'Feature Set'], 
        columns='Target', 
        aggfunc='mean'
    )
    sns.heatmap(pivot_rmse, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax1)
    ax1.set_title('RMSE (Test) Karşılaştırması')
    
    # R² karşılaştırması
    ax2 = axes[0, 1]
    pivot_r2 = results_df.pivot_table(
        values='R² (Test)', 
        index=['Model', 'Feature Set'], 
        columns='Target', 
        aggfunc='mean'
    )
    sns.heatmap(pivot_r2, annot=True, fmt='.4f', cmap='YlGn', ax=ax2)
    ax2.set_title('R² (Test) Karşılaştırması')
    
    # Direction Accuracy karşılaştırması
    ax3 = axes[1, 0]
    pivot_dir = results_df.pivot_table(
        values='Direction Accuracy (Test)', 
        index=['Model', 'Feature Set'], 
        columns='Target', 
        aggfunc='mean'
    )
    sns.heatmap(pivot_dir, annot=True, fmt='.4f', cmap='Blues', ax=ax3)
    ax3.set_title('Direction Accuracy (Test) Karşılaştırması')
    
    # Model bazında ortalama performans
    ax4 = axes[1, 1]
    model_performance = results_df.groupby('Model').agg({
        'RMSE (Test)': 'mean',
        'R² (Test)': 'mean',
        'Direction Accuracy (Test)': 'mean'
    })
    model_performance.plot(kind='bar', ax=ax4, rot=0)
    ax4.set_title('Model Bazında Ortalama Performans')
    ax4.set_ylabel('Skor')
    ax4.legend(loc='best')
    
    plt.tight_layout()
    plt.savefig(PROCESSED_DIR / 'model_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n✅ Görselleştirmeler kaydedildi: {PROCESSED_DIR / 'model_comparison.png'}")
